# 비교 실험
## 작업 1 파라미터 수 비교
## 작업 2 편집 성능 비교

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

work_dir = '/content/drive/MyDrive/machine_learning_hw10/IP2P_LoRA_FT'
sys.path.append(work_dir)

model_dir = os.path.join(work_dir, 'saved_models')
comparison_results_dir = os.path.join(work_dir, 'comparison_results')
os.makedirs(comparison_results_dir, exist_ok=True)

LORA_METHOD1_PATH = os.path.join(model_dir, "lora_method1.pt")
LORA_METHOD2_PATH = os.path.join(model_dir, "lora_method2.pt") # 본인이 선택한 lora_method2 의 가중치 파일 이름으로 변경

In [ ]:
!pip install -q diffusers transformers accelerate datasets pillow matplotlib

In [ ]:
from lora_utils import LoRALinearLayer, LoRALayer, apply_lora_to_unet_full, apply_lora_to_unet_selective

import torch
import torch.nn as nn
from diffusers import StableDiffusionInstructPix2PixPipeline, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel
from datasets import load_dataset
import matplotlib.pyplot as plt
import random

device = torch.device("cuda")

In [ ]:
# 설정
MODEL_ID = "timbrooks/instruct-pix2pix"
DATASET_NAME = "instruction-tuning-sd/cartoonization"
LORA_RANK = 8
LORA_ALPHA = 8
SELECTED_BLOCKS = ["up", "mid"] # 본인이 선택한 lora_method2 의 적용 블록으로 변경
SEED = 42
TRAIN_TEST_SPLIT = 0.96

## 테스트 샘플 준비

In [ ]:
# 테스트 데이터 로드
dataset = load_dataset(DATASET_NAME)
total_samples = len(dataset['train'])
num_train = int(total_samples * TRAIN_TEST_SPLIT)

shuffled = dataset['train'].shuffle(seed=SEED)
test_dataset = shuffled.select(range(num_train, total_samples))

# 랜덤 샘플 선택
random.seed()
random_idx = random.randint(0, len(test_dataset) - 1)
test_sample = test_dataset[random_idx]

test_image = test_sample['original_image']
test_prompt = test_sample['edit_prompt']
ground_truth = test_sample['cartoonized_image']

print(f"Test index: {random_idx}")
print(f"Prompt: {test_prompt}")

---
# 작업 1: 파라미터 수 비교

In [ ]:
# UNet 로드 및 파라미터 계산
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
full_params = sum(p.numel() for p in unet.parameters())

# LoRA 파라미터 계산 함수
def count_lora_params(unet, rank, target_modules=["to_q", "to_k", "to_v", "to_out.0"]):
    lora_params = 0
    def count_recursive(module, name=""):
        nonlocal lora_params
        for child_name, child_module in module.named_children():
            full_name = f"{name}.{child_name}" if name else child_name
            if isinstance(child_module, nn.Linear) and any(t in full_name for t in target_modules):
                lora_params += (child_module.in_features * rank) + (rank * child_module.out_features)
            else:
                count_recursive(child_module, full_name)
    count_recursive(unet)
    return lora_params

def count_lora_params_selective(unet, rank, selected_blocks, target_modules=["to_q", "to_k", "to_v", "to_out.0"]):
    lora_params = 0
    def should_apply(full_name):
        return any([("down_blocks" in full_name and "down" in selected_blocks),
                   ("mid_block" in full_name and "mid" in selected_blocks),
                   ("up_blocks" in full_name and "up" in selected_blocks)])

    def count_recursive(module, name=""):
        nonlocal lora_params
        for child_name, child_module in module.named_children():
            full_name = f"{name}.{child_name}" if name else child_name
            if isinstance(child_module, nn.Linear) and any(t in full_name for t in target_modules):
                if should_apply(full_name):
                    lora_params += (child_module.in_features * rank) + (rank * child_module.out_features)
            else:
                count_recursive(child_module, full_name)
    count_recursive(unet)
    return lora_params

lora1_params = count_lora_params(unet, LORA_RANK)
lora2_params = count_lora_params_selective(unet, LORA_RANK, SELECTED_BLOCKS)

print("="*70)
print("작업 4-1: 파라미터 수 비교")
print("="*70)
print(f"Full Fine-tuning (BFF):     {full_params:>12,} ({full_params/1e6:.1f}M)")
print(f"LoRA Method 1 (LFF):        {lora1_params:>12,} ({lora1_params/1e6:.2f}M)")
print(f"LoRA Method 2 (LPF):        {lora2_params:>12,} ({lora2_params/1e6:.2f}M)")
print(f"\nLFF efficiency: {full_params/lora1_params:.0f}x parameter reduction")
print(f"LPF efficiency: {full_params/lora2_params:.0f}x parameter reduction")
print("="*70)

del unet
torch.cuda.empty_cache()

In [ ]:
# 파라미터 비교 차트
methods = ['Full FT\n(BFF)', 'LoRA-1\n(LFF)', 'LoRA-2\n(LPF)']
params = [full_params, lora1_params, lora2_params]
colors = ['red', 'blue', 'green']

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar(methods, [p/1e6 for p in params], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Trainable Parameters (M)', fontsize=14, fontweight='bold')
ax.set_title('작업 4-1: Parameter Comparison', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--')

for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{p/1e6:.2f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(comparison_results_dir, '01_parameter_comparison.png'), dpi=150)
plt.show()

---
# 작업 2: 편집 성능 비교

In [ ]:
# Full Fine-tuning Model Inference
print("Loading Full FT model...")
full_pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "doa12/instructPix2Pix-cartoonization-full",
    torch_dtype=torch.float16,
    safety_checker=None
).to(device)

with torch.no_grad():
    full_output = full_pipe(
        test_prompt, image=test_image, num_inference_steps=20,
        image_guidance_scale=1.5, guidance_scale=7.0
    ).images[0]

del full_pipe
torch.cuda.empty_cache()
print("Full FT inference done")

In [ ]:
# LoRA Method 1 Inference
print("Loading LoRA Method 1...")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_layers_1, _ = apply_lora_to_unet_full(unet, rank=LORA_RANK, alpha=LORA_ALPHA)
lora_state = torch.load(LORA_METHOD1_PATH, map_location='cpu')

for i, lora_layer in enumerate(lora_layers_1):
    lora_layer.lora.lora_A.data = lora_state[f"lora_{i}_A"].to(device)
    lora_layer.lora.lora_B.data = lora_state[f"lora_{i}_B"].to(device)

pipe_method1 = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    MODEL_ID, unet=unet, text_encoder=text_encoder, vae=vae,
    torch_dtype=torch.float32, safety_checker=None
).to(device)

with torch.no_grad():
    output_method1 = pipe_method1(
        test_prompt, image=test_image, num_inference_steps=20,
        image_guidance_scale=1.5, guidance_scale=7.0
    ).images[0]

del pipe_method1, unet, vae, text_encoder
torch.cuda.empty_cache()
print("LoRA Method 1 inference done")

In [ ]:
# LoRA Method 2 Inference
print("Loading LoRA Method 2...")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_layers_2, _ = apply_lora_to_unet_selective(
    unet, rank=LORA_RANK, alpha=LORA_ALPHA, selected_blocks=SELECTED_BLOCKS
)
lora_state = torch.load(LORA_METHOD2_PATH, map_location='cpu')

for i, lora_layer in enumerate(lora_layers_2):
    lora_layer.lora.lora_A.data = lora_state[f"lora_{i}_A"].to(device)
    lora_layer.lora.lora_B.data = lora_state[f"lora_{i}_B"].to(device)

pipe_method2 = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    MODEL_ID, unet=unet, text_encoder=text_encoder, vae=vae,
    torch_dtype=torch.float32, safety_checker=None
).to(device)

with torch.no_grad():
    output_method2 = pipe_method2(
        test_prompt, image=test_image, num_inference_steps=20,
        image_guidance_scale=1.5, guidance_scale=7.0
    ).images[0]

del pipe_method2, unet, vae, text_encoder
torch.cuda.empty_cache()
print("LoRA Method 2 inference done")

In [ ]:
# 결과 비교 시각화
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(test_image)
axes[0, 0].set_title('Original Image', fontsize=14, weight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(full_output)
axes[0, 1].set_title('Full Fine-tuning (BFF)', fontsize=14, weight='bold', color='red')
axes[0, 1].axis('off')

axes[0, 2].imshow(ground_truth)
axes[0, 2].set_title('Ground Truth', fontsize=14, weight='bold')
axes[0, 2].axis('off')

axes[1, 0].axis('off')

axes[1, 1].imshow(output_method1)
axes[1, 1].set_title('LoRA Method 1 (LFF)', fontsize=14, weight='bold', color='blue')
axes[1, 1].axis('off')

axes[1, 2].imshow(output_method2)
axes[1, 2].set_title('LoRA Method 2 (LPF)', fontsize=14, weight='bold', color='green')
axes[1, 2].axis('off')

plt.suptitle(f'작업 4-2: 편집 성능 비교\nPrompt: "{test_prompt}"',
             fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.savefig(os.path.join(comparison_results_dir, '02_editing_comparison.png'), dpi=150)
plt.show()

print(f"\n결과 저장 완료: {comparison_results_dir}")

In [ ]:
# 최종 요약
print("\n" + "="*80)
print("비교 실험 완료")
print("="*80)
print(f"\n[작업 4-1] 파라미터 비교:")
print(f"  BFF: {full_params/1e6:.1f}M | LFF: {lora1_params/1e6:.2f}M | LPF: {lora2_params/1e6:.2f}M")
print(f"\n[작업 4-2] 편집 성능 비교:")
print(f"  Test index: {random_idx} | Prompt: {test_prompt}")
print(f"\n결과 저장 위치: {comparison_results_dir}/")
print("="*80)